# ICD Code Validation for Target Conditions

This notebook validates ICD codes for the 4 target diseases by merging:
- `diagnoses_icd.csv(.gz)`
- `d_icd_diagnoses.csv(.gz)`

It prints all matched ICD-9/ICD-10 codes for:
- sepsis
- heart failure
- chronic kidney disease (CKD)
- diabetes

and highlights codes detected by description keywords but not by the current prefix-only rules.

In [7]:
from pathlib import Path

import re



import pandas as pd



RAW_DIR = Path('/home/ubuntu/condition-aware_risk_CDSS/data/raw')

OUT_DIR = Path('/home/ubuntu/condition-aware_risk_CDSS/data/processed/icd_review')

OUT_DIR.mkdir(parents=True, exist_ok=True)



def resolve_file(base_name: str, search_dirs=None) -> Path | None:

    if search_dirs is None:

        search_dirs = [RAW_DIR]

    for d in search_dirs:

        gz = d / f'{base_name}.csv.gz'

        csv = d / f'{base_name}.csv'

        if gz.exists():

            return gz

        if csv.exists():

            return csv

    return None



diagnoses_path = resolve_file('diagnoses_icd')

if diagnoses_path is None:

    raise FileNotFoundError(f'Could not find diagnoses_icd.csv(.gz) in {RAW_DIR}')



dict_path = resolve_file('d_icd_diagnoses', search_dirs=[RAW_DIR, RAW_DIR.parent / 'csv'])



diagnoses = pd.read_csv(diagnoses_path, compression='infer')



if dict_path is not None:

    d_icd = pd.read_csv(dict_path, compression='infer')

    print('Loaded:', dict_path.name, d_icd.shape)

else:

    d_icd = diagnoses[['icd_code', 'icd_version']].drop_duplicates().copy()

    d_icd['long_title'] = ''

    print('WARNING: d_icd_diagnoses.csv(.gz) not found. Running with code-only matching (no title cross-check).')



print('Loaded:', diagnoses_path.name, diagnoses.shape)


Loaded: d_icd_diagnoses.csv.gz (112107, 3)
Loaded: diagnoses_icd.csv.gz (6364488, 5)


In [8]:
# Clean + merge diagnosis occurrences with ICD dictionary
key_cols = ['icd_code', 'icd_version']

for df in (diagnoses, d_icd):
    df['icd_code'] = df['icd_code'].astype(str).str.strip().str.upper()
    df['icd_version'] = pd.to_numeric(df['icd_version'], errors='coerce').astype('Int64')

d_icd['long_title'] = d_icd['long_title'].astype(str).fillna('')
d_icd['long_title_norm'] = d_icd['long_title'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()

merged = diagnoses.merge(
    d_icd[key_cols + ['long_title', 'long_title_norm']],
    on=key_cols,
    how='left'
)

code_usage = (
    diagnoses.groupby(key_cols, dropna=False)
    .agg(
        n_rows=('icd_code', 'size'),
        n_subjects=('subject_id', 'nunique'),
        n_hadm=('hadm_id', 'nunique')
    )
    .reset_index()
)

code_dict = d_icd[key_cols + ['long_title', 'long_title_norm']].drop_duplicates()
code_catalog = code_dict.merge(code_usage, on=key_cols, how='left').fillna({'n_rows': 0, 'n_subjects': 0, 'n_hadm': 0})

print('Merged diagnosis rows:', merged.shape)
print('Unique code catalog rows:', code_catalog.shape)

Merged diagnosis rows: (6364488, 7)
Unique code catalog rows: (112107, 7)


In [9]:
# Current prefix rules used in the pipeline
prefix_rules = {
    'diabetes': {
        9: ['250'],
        10: ['E08', 'E09', 'E10', 'E11', 'E13'],
    },
    'ckd': {
        9: ['585'],
        10: ['N18'],
    },
    'heart_failure': {
        9: ['428'],
        10: ['I50'],
    },
    'sepsis': {
        9: ['038', '99591', '99592', '78552'],
        10: ['A40', 'A41', 'R652'],
    },
}

# Additional description keyword rules for cross-checking
title_patterns = {
    'diabetes': re.compile(r'diabet', flags=re.IGNORECASE),
    'ckd': re.compile(r'chronic kidney disease|chronic renal|end stage renal|\bckd\b', flags=re.IGNORECASE),
    'heart_failure': re.compile(r'heart failure|congestive heart failure|\bchf\b', flags=re.IGNORECASE),
    'sepsis': re.compile(r'sepsis|septic shock|severe sepsis', flags=re.IGNORECASE),
}

def matches_prefix(row, condition):
    v = row['icd_version']
    code = row['icd_code']
    if pd.isna(v) or pd.isna(code):
        return False
    v = int(v)
    prefixes = prefix_rules.get(condition, {}).get(v, [])
    return any(str(code).startswith(p) for p in prefixes)

def matches_title(row, condition):
    txt = str(row.get('long_title_norm', '') or '')
    pattern = title_patterns[condition]
    return bool(pattern.search(txt))

In [10]:
all_condition_codes = []

for condition in ['sepsis', 'heart_failure', 'ckd', 'diabetes']:
    temp = code_catalog.copy()
    temp['match_prefix'] = temp.apply(lambda r: matches_prefix(r, condition), axis=1)
    temp['match_title'] = temp.apply(lambda r: matches_title(r, condition), axis=1)
    temp = temp[(temp['match_prefix']) | (temp['match_title'])].copy()

    temp['condition'] = condition
    temp['match_source'] = temp.apply(
        lambda r: 'prefix+title' if r['match_prefix'] and r['match_title']
        else ('prefix_only' if r['match_prefix'] else 'title_only'),
        axis=1,
    )

    temp = temp.sort_values(['icd_version', 'icd_code'])
    cols = [
        'condition', 'icd_version', 'icd_code', 'long_title',
        'match_source', 'n_rows', 'n_subjects', 'n_hadm'
    ]
    temp = temp[cols]

    print(f'\n=== {condition.upper()} ===')
    print(f'Total matched codes: {temp.shape[0]}')
    print(temp[['icd_version', 'icd_code', 'long_title', 'match_source']].to_string(index=False))

    all_condition_codes.append(temp)

all_codes_df = pd.concat(all_condition_codes, ignore_index=True)
all_codes_df.to_csv(OUT_DIR / 'icd_codes_for_4_conditions_full.csv', index=False)

summary = (
    all_codes_df.groupby(['condition', 'match_source'])['icd_code']
    .count()
    .rename('n_codes')
    .reset_index()
)
summary.to_csv(OUT_DIR / 'icd_codes_for_4_conditions_summary.csv', index=False)

print('\nSaved full code list to:', OUT_DIR / 'icd_codes_for_4_conditions_full.csv')
print('Saved summary to:', OUT_DIR / 'icd_codes_for_4_conditions_summary.csv')


=== SEPSIS ===
Total matched codes: 84
 icd_version icd_code                                                            long_title match_source
           9     0380                                              Streptococcal septicemia  prefix_only
           9    03810                                Staphylococcal septicemia, unspecified  prefix_only
           9    03811              Methicillin susceptible Staphylococcus aureus septicemia  prefix_only
           9    03812                Methicillin resistant Staphylococcus aureus septicemia  prefix_only
           9    03819                                       Other staphylococcal septicemia  prefix_only
           9     0382         Pneumococcal septicemia [Streptococcus pneumoniae septicemia]  prefix_only
           9     0383                                           Septicemia due to anaerobes  prefix_only
           9    03840                 Septicemia due to gram-negative organism, unspecified  prefix_only
           9   

In [11]:
# Review candidates detected by title keywords but NOT by current prefix rules
title_only = all_codes_df[all_codes_df['match_source'] == 'title_only'].copy()

if title_only.empty:
    print('No title-only candidates found. Current prefix rules already cover matched descriptions.')
else:
    print('Title-only candidates for review (possible codebook expansion):')
    print(title_only[['condition', 'icd_version', 'icd_code', 'long_title', 'n_rows']].to_string(index=False))

title_only.to_csv(OUT_DIR / 'icd_codes_title_only_candidates.csv', index=False)
print('Saved candidate review file to:', OUT_DIR / 'icd_codes_title_only_candidates.csv')

Title-only candidates for review (possible codebook expansion):
    condition  icd_version icd_code                                                                                                                                                      long_title  n_rows
       sepsis            9    67020                                                                                           Puerperal sepsis, unspecified as to episode of care or not applicable     0.0
       sepsis            9    67022                                                                                            Puerperal sepsis, delivered, with mention of postpartum complication    10.0
       sepsis            9    67024                                                                                                          Puerperal sepsis, postpartum condition or complication     8.0
       sepsis            9    77181                                                                                     